# One-Time Setup

## 1.挂载Google Drive

In [ ]:
# 挂载Google Drive
from google.colab import drive
import os

# 挂载 Google Drive
drive.mount('/content/drive')

# ── 全局路径变量（所有 cell 均可直接使用）──────────────────
LAB_DIR    = "/content/drive/MyDrive/HeteroInfer_Lab"
MODEL_DIR  = f"{LAB_DIR}/models"
FIGURE_DIR = f"{LAB_DIR}/figures"

# 目录不存在则自动创建，已存在则跳过
os.makedirs(MODEL_DIR,  exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

print(f"✅ Google Drive mounted")
print(f"   MODEL_DIR  → {MODEL_DIR}")
print(f"   FIGURE_DIR → {FIGURE_DIR}")

## 2.Install Dependencies

In [ ]:
# 安装 Experiment 001 所需的最小依赖
# transformers：加载 Gemma 模型与 tokenizer
# accelerate：辅助模型自动放置到 GPU
# sentencepiece：部分 tokenizer 需要
!pip install -q transformers accelerate sentencepiece huggingface_hub

## 3.Authentication

In [ ]:
# HuggingFace
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))
print("✅ HF login success")

## 4.Load Model

In [ ]:
# 导入必要库
# torch 用于控制 GPU 与推理精度
# AutoTokenizer / AutoModelForCausalLM 用于从 Hugging Face 加载模型
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download


# 先使用小模型，保证 T4 16GB 显存下能稳定跑通
# 如果这个模型需要 Hugging Face 授权，我们再处理登录
model_id = "google/gemma-2-2b-it"
model_name = model_id.split("/")[-1]        # "gemma-2-2b-it"
local_path = f"{MODEL_DIR}/{model_name}"      # 最终存放路径

# ── 检测本地缓存 ────────────────────────────────────────
if os.path.isdir(local_path) and os.listdir(local_path):
    print(f"✅ Found local model at {local_path}, skipping download.")
else:
    print(f"⬇️  Downloading {model_id} → {local_path} ...")
    snapshot_download(
        repo_id   = model_id,
        local_dir = local_path,
    )
    print("✅ Download complete.")

# ── 从本地路径加载 ───────────────────────────────────────
# 加载 tokenizer：负责把文本转成 token，也负责把输出 token 解码回文本
tokenizer = AutoTokenizer.from_pretrained(local_path)

# 加载模型
# torch_dtype=torch.float16：使用 FP16 降低显存占用
# device_map="auto"：让 transformers/accelerate 自动把模型放到 GPU
model = AutoModelForCausalLM.from_pretrained(
    local_path,
    torch_dtype = torch.float16,
    device_map  = "auto",
)

print(f"✅ Model loaded: {model_id}")
print(f"   Device: {model.device}")

# Baseline Environment Check

In [ ]:
import torch
import transformers
import platform


#GPU检测
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("CUDA Version:", torch.version.cuda)
else:
    print("No GPU found")

# 软件及框架版本
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)


# Baseline Profiling

In [ ]:
# 清空显存
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

## 1.Gemma Baseline Profiling

In [ ]:
import torch
import time
import gc

# ═══════════════════════════════════════════════════════════
# 工具函数区
# ═══════════════════════════════════════════════════════════

def get_gpu_memory_mb():
    """
    读取当前GPU已分配显存（MB）。

    注意：这是"已分配"而非"已使用"。PyTorch的显存分配器会提前
    申请一块显存池，所以这个数字反映的是PyTorch向GPU申请了多少，
    不是操作系统层面显示的显存占用。
    """
    return torch.cuda.memory_allocated() / 1024**2  # bytes → MB


def estimate_kv_cache_mb(model, seq_len, batch_size=1):
    """
    从模型结构参数出发，理论估算KV Cache的显存大小。

    KV Cache是什么：
        在autoregressive推理中，每次decode一个新token时，
        模型需要访问所有之前token的Key和Value矩阵。
        KV Cache就是把这些中间结果缓存下来，避免重复计算。

    公式推导：
        每个token在每一层会产生：
            - 1个Key向量：形状 = (num_kv_heads, head_dim)
            - 1个Value向量：形状 = (num_kv_heads, head_dim)

        整个序列的KV Cache大小 =
            2          （K和V各一份）
            × num_layers      （每层都有独立的KV）
            × num_kv_heads    （GQA/MQA模型的KV head数，可能小于Q head数）
            × head_dim        （每个head的维度）
            × seq_len         （已处理的token数量）
            × batch_size      （并发请求数）
            × bytes_per_elem  （float16 = 2字节）

    参数说明：
        model   : 已加载的HuggingFace模型
        seq_len : prefill长度 + 已生成token数（总序列长度）
        batch_size: 并行batch数，我们实验固定为1

    返回：
        估算的KV Cache大小（MB）

    局限性：
        这是理论下界。实际显存占用还包括：
        - PyTorch的显存碎片
        - 激活值（activations）
        - attention score矩阵（如果不用flash attention）
        所以实测peak_mem会高于这个估算值。
    """
    cfg = model.config

    # 总层数：Transformer的decoder层数（Gemma 2-2B = 26层）
    num_layers = cfg.num_hidden_layers

    # KV head数：现代模型常用GQA（分组查询注意力），KV head数 < Q head数
    # getattr的fallback处理：如果模型config没有num_key_value_heads字段，
    # 就退化为标准MHA（KV head数 = Q head数）
    num_kv_heads = getattr(cfg, 'num_key_value_heads', cfg.num_attention_heads)

    # 每个head的维度：Gemma系列有显式的head_dim字段；
    # 如果没有，用 hidden_size ÷ num_attention_heads 计算
    head_dim = cfg.head_dim if hasattr(cfg, 'head_dim') else cfg.hidden_size // cfg.num_attention_heads

    bytes_per_elem = 2  # float16精度，每个元素占2字节

    # 套公式
    total_bytes = (2 * num_layers * num_kv_heads * head_dim
                   * seq_len * batch_size * bytes_per_elem)

    return total_bytes / 1024**2  # bytes → MB


# ═══════════════════════════════════════════════════════════
# 核心测量函数
# ═══════════════════════════════════════════════════════════

def measure_single(prompt_len, gen_len, model, tokenizer, device):
    """
    对一组(prompt_len, gen_len)参数进行单次完整的性能测量。

    测量流程：
        1. 构造指定长度的输入prompt
        2. 跑一次完整的prefill，计时得到TTFT
        3. 逐token跑gen_len次decode，每次计时得到单步TPOT
        4. 读取峰值显存，估算KV Cache
        5. 清理显存，返回结果字典

    为什么prefill和decode要分开测量：
        - prefill是"并行处理整个输入序列"，计算密集，受限于GPU算力
        - decode是"逐token串行生成"，每步只处理1个token，受限于显存带宽
        这两个阶段的性能瓶颈不同，必须分开分析，这是LLM推理研究的核心问题。
    """

    # ── Step 1: 构造输入 ──────────────────────────────────
    # 用一个固定的英文句子重复填充，确保token分布稳定
    # prompt_len // 10 + 1 是为了保证重复次数足够多，能截断到目标长度
    base = "The quick brown fox jumps over the lazy dog. "
    prompt = base * (prompt_len // 10 + 1)

    # tokenizer负责把文本转成token id序列
    # truncation=True + max_length=prompt_len：截断到目标长度
    # return_tensors="pt"：返回PyTorch tensor
    # .to(device)：把tensor搬到GPU上
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=prompt_len
    ).to(device)

    # 记录实际token数（可能因分词边界略小于prompt_len）
    actual_prompt_len = inputs.input_ids.shape[1]

    # ── Step 2: 重置显存统计 ──────────────────────────────
    # synchronize()：等待GPU上所有异步操作完成，确保计时起点干净
    # reset_peak_memory_stats()：清零PyTorch的峰值显存记录器，
    #   这样后面读到的peak值只反映本次实验的最高点
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    mem_before = get_gpu_memory_mb()  # 记录基准显存（当前已占用）

    # ── Step 3: Prefill阶段 → 测量TTFT ───────────────────
    # TTFT（Time To First Token）：从输入prompt到生成第一个token的延迟
    # 这个阶段模型并行处理所有输入token，是计算密集型操作
    #
    # use_cache=True：让模型保存KV Cache，供decode阶段复用
    # torch.no_grad()：推理阶段不需要计算梯度，节省显存和计算
    t0 = time.perf_counter()  # perf_counter比time.time()精度更高
    with torch.no_grad():
        outputs = model(input_ids=inputs.input_ids, use_cache=True)
    torch.cuda.synchronize()  # 必须sync，否则GPU还没跑完就记时了
    ttft = (time.perf_counter() - t0) * 1000  # 转换为毫秒

    # 从prefill输出中取出两个关键结果：
    # past_key_values：缓存好的KV矩阵，形状=(num_layers, 2, batch, heads, seq, dim)
    # next_token：prefill阶段预测的第一个输出token（取logits最大值的index）
    past_kv = outputs.past_key_values
    next_token = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)
    # logits形状：(batch=1, seq_len, vocab_size)
    # [:, -1, :]取最后一个位置的预测；argmax取概率最大的token id
    # keepdim=True保持(1,1)的形状，作为下一步的输入

    # ── Step 4: Decode阶段 → 测量TPOT ────────────────────
    # TPOT（Time Per Output Token）：每生成一个新token的平均耗时
    # decode是串行的：每步输入1个token，利用KV Cache避免重算历史
    decode_times = []

    for _ in range(gen_len):
        t1 = time.perf_counter()
        with torch.no_grad():
            # 每次只输入最新生成的那1个token
            # past_key_values传入缓存，模型不需要重新处理历史token
            out = model(
                input_ids=next_token,
                past_key_values=past_kv,
                use_cache=True
            )
        torch.cuda.synchronize()
        decode_times.append((time.perf_counter() - t1) * 1000)

        # 更新KV Cache和下一个token（自回归：用刚生成的token预测下一个）
        past_kv = out.past_key_values
        next_token = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)

    # ── Step 5: 汇总指标 ─────────────────────────────────
    tpot = sum(decode_times) / len(decode_times)  # 平均每token耗时(ms)
    tokens_s = 1000.0 / tpot                       # 吞吐率：tokens/秒

    # 峰值显存：本次实验从reset到现在的最高显存占用点
    peak_mem = torch.cuda.max_memory_allocated() / 1024**2

    # KV Cache理论估算：传入(prefill长度 + 已生成长度)作为总序列长度
    kv_est = estimate_kv_cache_mb(model, actual_prompt_len + gen_len)

    # ── Step 6: 清理显存 ─────────────────────────────────
    # KV Cache在显存里会持续占用，如果不手动清理，
    # 下一次实验的peak_mem读数会被上次的KV Cache污染
    del past_kv, outputs, out
    gc.collect()               # 触发Python垃圾回收
    torch.cuda.empty_cache()   # 把PyTorch缓存的显存还给GPU驱动

    return {
        "prompt_len":  actual_prompt_len,   # 实际prompt token数
        "gen_len":     gen_len,             # 生成token数
        "ttft_ms":     round(ttft, 2),      # 首token延迟(ms)
        "tpot_ms":     round(tpot, 2),      # 平均单token解码延迟(ms)
        "tokens_s":    round(tokens_s, 1),  # 解码吞吐率(tokens/s)
        "peak_mem_mb": round(peak_mem, 1),  # 峰值显存(MB)
        "kv_est_mb":   round(kv_est, 2),    # KV Cache估算大小(MB)
    }


# ═══════════════════════════════════════════════════════════
# 实验主循环：扫描(prompt_len × gen_len)参数矩阵
# ═══════════════════════════════════════════════════════════

results = []
device = next(model.parameters()).device

# 实验参数矩阵：
# prompt_lengths：控制prefill阶段的输入长度，观察TTFT随输入增长的趋势
# gen_lengths   ：控制decode阶段的生成长度，观察TPOT和显存的变化
# 共4×3=12组实验
prompt_lengths = [64, 128, 256, 512]
gen_lengths    = [32, 64, 128]

# ── Warm-up：多组预热，覆盖短/中/长序列的所有CUDA kernel路径 ──
#
# 为什么要多组：
#   PyTorch的CUDA kernel是"懒编译"的——某个(序列长度, batch_size)组合
#   第一次被执行时才编译对应的kernel，这个编译耗时会污染计时结果。
#   单次warm-up只能热身一种序列长度的kernel路径，
#   所以需要覆盖实验矩阵中所有会出现的序列长度范围。
#
# warm-up结果不记录，只是为了触发kernel编译和GPU状态稳定。
# 每组跑完后清理显存，避免影响后续实验的peak memory统计。

WARMUP_CONFIGS = [
    (64,  32),   # 短序列
    (256, 64),   # 中序列
    (512, 32),   # 长序列prefill
]

print("🔥 Warming up GPU...")
for wp, wg in WARMUP_CONFIGS:
    _ = measure_single(wp, wg, model, tokenizer, device)
    print(f"   ✓ warm-up ({wp}, {wg}) done")
print("✅ Warm-up done\n")


# 正式实验
print("🚀 Experiment 001: Gemma Baseline Profiling")
print(f"{'prompt':>8} {'gen':>6} {'TTFT(ms)':>10} {'TPOT(ms)':>10} "
      f"{'tok/s':>8} {'peak(MB)':>10} {'KV_est(MB)':>12}")
print("-" * 70)

for pl in prompt_lengths:
    for gl in gen_lengths:
        r = measure_single(pl, gl, model, tokenizer, device)
        results.append(r)
        # 格式化输出，右对齐便于对比
        print(f"{r['prompt_len']:>8} {r['gen_len']:>6} {r['ttft_ms']:>10} "
              f"{r['tpot_ms']:>10} {r['tokens_s']:>8} "
              f"{r['peak_mem_mb']:>10} {r['kv_est_mb']:>12}")

print(f"\n✅ Done. Total experiments: {len(results)}")
# results列表保存所有结果，下一个cell做可视化时直接用

In [ ]:
import pandas as pd
import os

# ── 定义结果保存目录 ────────────────────────────────────────────
# 实验结果统一存放在 results/exp001/ 下，与模型、图表分开管理
RESULT_DIR = "/content/drive/MyDrive/HeteroInfer_Lab/results/exp001"
os.makedirs(RESULT_DIR, exist_ok=True)

# ── 将 results 列表转为 DataFrame ─────────────────────────────
df = pd.DataFrame(results)

# ── 保存为 CSV ────────────────────────────────────────────────
csv_path = f"{RESULT_DIR}/gemma_baseline_profiling.csv"
df.to_csv(csv_path, index=False)

print(f"✅ CSV 已保存：{csv_path}")
print(f"   共 {len(df)} 条记录，列：{list(df.columns)}")
print()
print(df.to_string(index=False))

## 2.Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import os

# ── 定义图表保存目录（与 CSV 结果同层级）──────────────────────────
EXP_FIGURE_DIR = "/content/drive/MyDrive/HeteroInfer_Lab/results/exp001/figures"
os.makedirs(EXP_FIGURE_DIR, exist_ok=True)
# ── 把results列表转成DataFrame，方便按列筛选 ──
df = pd.DataFrame(results)

# 排除冷启动的第一条数据（prompt_len=64, gen_len=32的第一次运行）
# 判断标准：TTFT异常大（>500ms视为冷启动）
df_clean = df[df['ttft_ms'] < 500].copy()

# 颜色方案：不同gen_len用不同颜色区分
gen_len_list = sorted(df_clean['gen_len'].unique())   # [32, 64, 128]
colors = ['#4C9BE8', '#F5A623', '#7ED321']            # 蓝/橙/绿

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Experiment 001: Gemma 2-2B Baseline Profiling\n(Tesla T4, FP16, Colab Free)',
             fontsize=14, fontweight='bold')

# ── 图1：TTFT vs Prompt Length ────────────────────────────
# 核心问题：prefill延迟如何随输入长度增长？
# 预期：线性增长（因为prefill的计算量∝seq_len）
ax1 = axes[0, 0]
for i, gl in enumerate(gen_len_list):
    subset = df_clean[df_clean['gen_len'] == gl].sort_values('prompt_len')
    ax1.plot(subset['prompt_len'], subset['ttft_ms'],
             marker='o', color=colors[i], label=f'gen={gl}', linewidth=2)
ax1.set_xlabel('Prompt Length (tokens)')
ax1.set_ylabel('TTFT (ms)')
ax1.set_title('① TTFT vs Prompt Length\n(Prefill Latency)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xscale('log', base=2)
ax1.set_xticks([64, 128, 256, 512])
ax1.xaxis.set_major_formatter(ticker.ScalarFormatter())
# 标注：如果曲线是直线，说明prefill是O(n)，符合理论

# ── 图2：TPOT vs Prompt Length ────────────────────────────
# 核心问题：decode速度是否受prompt长度影响？
# 预期：应该相对平稳（因为每步decode只处理1个token）
# 如果随prompt增大而变慢，说明KV Cache读取开销在增加
ax2 = axes[0, 1]
for i, gl in enumerate(gen_len_list):
    subset = df_clean[df_clean['gen_len'] == gl].sort_values('prompt_len')
    ax2.plot(subset['prompt_len'], subset['tpot_ms'],
             marker='s', color=colors[i], label=f'gen={gl}', linewidth=2)
ax2.set_xlabel('Prompt Length (tokens)')
ax2.set_ylabel('TPOT (ms)')
ax2.set_title('② TPOT vs Prompt Length\n(Decode Latency per Token)')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xscale('log', base=2)
ax2.set_xticks([64, 128, 256, 512])
ax2.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax2.set_ylim(0, 100)  # 固定Y轴范围，便于观察波动幅度

# ── 图3：Peak Memory vs Prompt Length ────────────────────
# 核心问题：显存主要被什么占用？
# 预期：随prompt增大而线性增长（更长的KV Cache）
# 关键观察：不同gen_len的曲线是否重合？（重合说明gen阶段追加的KV可忽略）
ax3 = axes[1, 0]
for i, gl in enumerate(gen_len_list):
    subset = df_clean[df_clean['gen_len'] == gl].sort_values('prompt_len')
    ax3.plot(subset['prompt_len'], subset['peak_mem_mb'],
             marker='^', color=colors[i], label=f'gen={gl}', linewidth=2)
# 画一条水平虚线标注T4总显存（16GB），直观显示还有多少余量
ax3.axhline(y=16*1024, color='red', linestyle='--', alpha=0.5, label='T4 limit (16GB)')
ax3.set_xlabel('Prompt Length (tokens)')
ax3.set_ylabel('Peak GPU Memory (MB)')
ax3.set_title('③ Peak Memory vs Prompt Length')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_xscale('log', base=2)
ax3.set_xticks([64, 128, 256, 512])
ax3.xaxis.set_major_formatter(ticker.ScalarFormatter())

# ── 图④：KV Cache 估算大小 vs Prompt Length ──────────────
# 修改原因：
#   1. 原版颜色只有3个但prompt_len有4个值，导致两条线撞色
#   2. X轴改为prompt_len，与图①②③保持一致，便于对比
#   3. 按gen_len分组，直观展示KV Cache主要由prompt长度决定

ax4 = axes[1, 1]

# 4个颜色对应4种gen_len，与图①②③的colors含义不同，
# 这里重新定义避免混淆
colors_gen = ['#4C9BE8', '#F5A623', '#7ED321']  # gen=32/64/128

for i, gl in enumerate(gen_len_list):  # gen_len_list = [32, 64, 128]
    subset = df_clean[df_clean['gen_len'] == gl].sort_values('prompt_len')
    ax4.plot(subset['prompt_len'], subset['kv_est_mb'],
             marker='D', color=colors_gen[i], label=f'gen={gl}', linewidth=2)

ax4.set_xlabel('Prompt Length (tokens)')
ax4.set_ylabel('KV Cache Estimated Size (MB)')
ax4.set_title('④ KV Cache Size vs Prompt Length\n(by Gen Length)')
ax4.set_xscale('log', base=2)
ax4.set_xticks([64, 128, 256, 512])
ax4.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax4.legend()
ax4.grid(True, alpha=0.3)


plt.tight_layout()
plt.savefig(f"{EXP_FIGURE_DIR}/exp001_profiling_results.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ 图表已保存：{EXP_FIGURE_DIR}/exp001_profiling_results.png")

## 3. V2 Profiling：改进测量 + 扩展序列长度

改进点：
1. `measure_single_v2`：直接遍历 `past_key_values` 中的 K/V tensors，得到更干净的 KV Cache 实测值（`kv_pkv_prefill_mb` / `kv_pkv_final_mb`）
2. 每组参数重复 3 次取均值，消除 TPOT 抖动
3. 扩展序列长度至 1024 / 2048，观察 KV Cache 从"可忽略"变为"显存主要占用者"的临界点

In [ ]:
import torch
import time
import gc

# ═══════════════════════════════════════════════════════════
# measure_single_v2：在 v1 基础上新增 past_key_values KV Cache 实测
# 改动：不再使用显存差分，而是直接统计 outputs.past_key_values
#       中 K/V tensor 的 numel × element_size。
# 原理：past_key_values 是模型实际保留下来供 decode 复用的 KV Cache，
#       避免把 PyTorch allocator、activation、临时 buffer 混入 KV 统计。
# ═══════════════════════════════════════════════════════════

def kv_cache_size_from_past_key_values_mb(past_key_values):
    """
    直接从 HuggingFace 返回的 past_key_values 统计 KV Cache tensor 大小。

    兼容两类常见结构：
    - legacy cache: tuple(layer_cache)，每层通常是 (key, value)
    - DynamicCache: 优先通过 to_legacy_cache() 转成 legacy 结构

    返回值只包含真正的 K/V tensor 字节数，不包含 PyTorch 预分配池、
    activation、attention 临时矩阵或 allocator 碎片。
    """
    candidates = [past_key_values]
    if hasattr(past_key_values, "to_legacy_cache"):
        try:
            candidates.append(past_key_values.to_legacy_cache())
        except Exception:
            pass

    seen_objects = set()
    seen_tensors = set()

    def walk(obj):
        if obj is None:
            return 0

        obj_id = id(obj)
        if obj_id in seen_objects:
            return 0
        seen_objects.add(obj_id)

        if torch.is_tensor(obj):
            tensor_id = id(obj)
            if tensor_id in seen_tensors:
                return 0
            seen_tensors.add(tensor_id)
            return obj.numel() * obj.element_size()

        if isinstance(obj, dict):
            return sum(walk(v) for v in obj.values())
        if isinstance(obj, (list, tuple)):
            return sum(walk(v) for v in obj)
        if isinstance(obj, (str, bytes, int, float, bool)):
            return 0

        total = 0
        for name in (
            "key_cache", "value_cache", "keys", "values",
            "k_cache", "v_cache", "cache", "caches", "layers",
        ):
            if hasattr(obj, name):
                try:
                    total += walk(getattr(obj, name))
                except Exception:
                    pass

        if total == 0 and hasattr(obj, "__dict__"):
            total += walk(vars(obj))

        return total

    total_bytes = sum(walk(c) for c in candidates)
    if total_bytes == 0:
        cache_type = type(past_key_values).__name__
        public_attrs = [a for a in dir(past_key_values) if not a.startswith("_")][:30]
        raise ValueError(
            f"past_key_values 中没有找到可统计的 KV Cache tensor；"
            f"cache_type={cache_type}, attrs={public_attrs}"
        )
    return round(total_bytes / 1024**2, 2)


def measure_single_v2(prompt_len, gen_len, model, tokenizer, device):

    # ── Step 1: 构造输入（与 v1 相同）──────────────────────
    base = "The quick brown fox jumps over the lazy dog. "
    prompt = base * (prompt_len // 10 + 1)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=prompt_len
    ).to(device)
    actual_prompt_len = inputs.input_ids.shape[1]

    # ── Step 2: 重置显存统计 ────────────────────────────────
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    # ── Step 3: Prefill 阶段 → 测量 TTFT + KV Cache 实测 ──
    # KV Cache 不再通过显存差分估算，而是直接读取 past_key_values。
    # 这样得到的是 K/V tensor 本身大小，杂质最少，便于和理论公式对比。

    t0 = time.perf_counter()
    with torch.no_grad():
        outputs = model(input_ids=inputs.input_ids, use_cache=True)
    torch.cuda.synchronize()
    ttft = (time.perf_counter() - t0) * 1000

    past_kv    = outputs.past_key_values
    kv_pkv_prefill_mb = kv_cache_size_from_past_key_values_mb(past_kv)
    next_token = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)

    # ── Step 4: Decode 阶段 → 测量 TPOT（与 v1 相同）───────
    decode_times = []
    for _ in range(gen_len):
        t1 = time.perf_counter()
        with torch.no_grad():
            out = model(
                input_ids=next_token,
                past_key_values=past_kv,
                use_cache=True
            )
        torch.cuda.synchronize()
        decode_times.append((time.perf_counter() - t1) * 1000)
        past_kv    = out.past_key_values
        next_token = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)

    # ── Step 5: 汇总指标 ────────────────────────────────────
    kv_pkv_final_mb = kv_cache_size_from_past_key_values_mb(past_kv)
    tpot     = sum(decode_times) / len(decode_times)
    tokens_s = 1000.0 / tpot
    peak_mem = torch.cuda.max_memory_allocated() / 1024**2
    kv_est   = estimate_kv_cache_mb(model, actual_prompt_len + gen_len)

    # ── Step 6: 清理显存 ────────────────────────────────────
    del past_kv, outputs, out
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "prompt_len":   actual_prompt_len,
        "gen_len":      gen_len,
        "ttft_ms":      round(ttft, 2),
        "tpot_ms":      round(tpot, 2),
        "tokens_s":     round(tokens_s, 1),
        "peak_mem_mb":  round(peak_mem, 1),
        "kv_est_mb":         round(kv_est, 2),      # 理论公式估算
        "kv_pkv_prefill_mb": kv_pkv_prefill_mb,      # prefill 后 PKV tensor 大小
        "kv_pkv_final_mb":   kv_pkv_final_mb,        # decode 后 PKV tensor 大小
    }

print("✅ measure_single_v2 已定义（past_key_values KV measurement）")

In [ ]:
import numpy as np

# ═══════════════════════════════════════════════════════════
# V2 实验主循环
# 改进1：每组参数重复 N_REPEAT 次取均值，降低单次测量噪声
# 改进2：扩展参数矩阵，覆盖 1024/2048 长序列
# ═══════════════════════════════════════════════════════════

N_REPEAT = 3   # 每组重复次数，取均值消除 TPOT 抖动

# ── 参数矩阵 ────────────────────────────────────────────────
# 原始矩阵（与 v1 相同，用于对比）
prompt_lengths_orig = [64, 128, 256, 512]
gen_lengths_orig    = [32, 64, 128]

# 扩展矩阵：观察长序列下 KV Cache 压力
# 注意：2048 接近 T4 16GB 显存上限，OOM 时自动降为 1536
prompt_lengths_ext = [512, 1024, 2048]
gen_lengths_ext    = [32, 128]

all_configs = (
    [(p, g) for p in prompt_lengths_orig for g in gen_lengths_orig] +
    [(p, g) for p in prompt_lengths_ext  for g in gen_lengths_ext]
)

# ── V2 Warm-up：覆盖短/中/长/超长序列所有 kernel 路径 ──────
# 相比 v1 新增 (1024, 32) 和 (2048, 32)，确保长序列 kernel 也已编译
WARMUP_CONFIGS_V2 = [
    (64,   32),
    (256,  64),
    (512,  32),
    (1024, 32),
    (2048, 32),
]

print("🔥 V2 Warming up GPU...")
for wp, wg in WARMUP_CONFIGS_V2:
    try:
        _ = measure_single_v2(wp, wg, model, tokenizer, device)
        print(f"   ✓ warm-up ({wp}, {wg}) done")
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print(f"   ⚠️  warm-up ({wp}, {wg}) OOM，跳过")
print("✅ V2 Warm-up done\n")

# ── 正式实验 ────────────────────────────────────────────────
results_v2 = []
device = next(model.parameters()).device

print("🚀 Experiment 001 V2: 重复测量 + 扩展序列长度")
print(f"{'prompt':>8} {'gen':>6} {'TTFT(ms)':>10} {'TPOT(ms)':>10} "
      f"{'tok/s':>8} {'peak(MB)':>10} {'kv_est(MB)':>11} "
      f"{'pkv_pre(MB)':>12} {'pkv_final(MB)':>13}")
print("-" * 108)

for pl, gl in all_configs:
    # 2048 可能 OOM，先尝试，OOM 则降为 1536
    actual_pl = pl
    if pl == 2048:
        try:
            _ = measure_single_v2(pl, gl, model, tokenizer, device)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print(f"   ⚠️  prompt=2048 OOM，自动降为 1536")
            actual_pl = 1536

    # 重复 N_REPEAT 次，收集原始数据
    raw = []
    for rep in range(N_REPEAT):
        try:
            r = measure_single_v2(actual_pl, gl, model, tokenizer, device)
            raw.append(r)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print(f"   ⚠️  ({actual_pl}, {gl}) rep{rep+1} OOM，跳过本次")
            break

    if not raw:
        continue

    # 对数值指标取均值，kv_est_mb 取第一次（理论值，每次相同）
    avg = {
        "prompt_len":   raw[0]["prompt_len"],
        "gen_len":      raw[0]["gen_len"],
        "ttft_ms":      round(np.mean([x["ttft_ms"]     for x in raw]), 2),
        "tpot_ms":      round(np.mean([x["tpot_ms"]     for x in raw]), 2),
        "tokens_s":     round(np.mean([x["tokens_s"]    for x in raw]), 1),
        "peak_mem_mb":  round(np.mean([x["peak_mem_mb"] for x in raw]), 1),
        "kv_est_mb":         raw[0]["kv_est_mb"],
        "kv_pkv_prefill_mb": round(np.mean([x["kv_pkv_prefill_mb"] for x in raw]), 2),
        "kv_pkv_final_mb":   round(np.mean([x["kv_pkv_final_mb"]   for x in raw]), 2),
    }
    results_v2.append(avg)

    print(f"{avg['prompt_len']:>8} {avg['gen_len']:>6} {avg['ttft_ms']:>10} "
          f"{avg['tpot_ms']:>10} {avg['tokens_s']:>8} "
          f"{avg['peak_mem_mb']:>10} {avg['kv_est_mb']:>11} "
          f"{avg['kv_pkv_prefill_mb']:>12} {avg['kv_pkv_final_mb']:>13}")

print(f"\n✅ V2 Done. Total experiments: {len(results_v2)}")

In [ ]:
import pandas as pd
import numpy as np
import os

# ── 保存路径检测：Kaggle 优先，否则用 Colab Google Drive 路径 ──
# 原因：同一份 notebook 在 Kaggle 和 Colab 两个环境均可运行
if os.path.exists('/kaggle/working'):
    RESULT_DIR_V2 = '/kaggle/working'
else:
    RESULT_DIR_V2 = "/content/drive/MyDrive/HeteroInfer_Lab/results/exp001"

os.makedirs(RESULT_DIR_V2, exist_ok=True)

# ── 保存 PKV CSV（_pkv 后缀，不覆盖原始 v2 文件）──────────────
df_v2 = pd.DataFrame(results_v2)
csv_v2_path = f"{RESULT_DIR_V2}/exp001_results_pkv.csv"
df_v2.to_csv(csv_v2_path, index=False)
print(f"✅ PKV CSV 已保存：{csv_v2_path}")
print(f"   共 {len(df_v2)} 条记录，列：{list(df_v2.columns)}\n")

# ── 新旧 TPOT 对比摘要 ───────────────────────────────────────
# 目的：确认 3 次取均值后 TPOT 抖动是否收窄
# 只比较两份数据共有的 (prompt_len, gen_len) 组合
df_v1 = pd.DataFrame(results)   # v1 原始结果（单次测量）

# 找出共有的参数组合
common_keys = set(zip(df_v1['prompt_len'], df_v1['gen_len'])) & \
              set(zip(df_v2['prompt_len'], df_v2['gen_len']))

print("=" * 60)
print("TPOT 新旧对比（相同参数组合）")
print(f"{'prompt':>8} {'gen':>6} {'v1 TPOT':>10} {'v2 TPOT':>10} {'变化':>8}")
print("-" * 60)

for pl, gl in sorted(common_keys):
    v1_tpot = df_v1[(df_v1['prompt_len'] == pl) & (df_v1['gen_len'] == gl)]['tpot_ms'].values[0]
    v2_tpot = df_v2[(df_v2['prompt_len'] == pl) & (df_v2['gen_len'] == gl)]['tpot_ms'].values[0]
    delta   = v2_tpot - v1_tpot
    flag    = "↑" if delta > 2 else ("↓" if delta < -2 else "≈")
    print(f"{pl:>8} {gl:>6} {v1_tpot:>10.2f} {v2_tpot:>10.2f} {flag:>5} {delta:+.2f}ms")

# 整体统计
v1_tpots = [df_v1[(df_v1['prompt_len']==pl) & (df_v1['gen_len']==gl)]['tpot_ms'].values[0]
            for pl, gl in common_keys]
v2_tpots = [df_v2[(df_v2['prompt_len']==pl) & (df_v2['gen_len']==gl)]['tpot_ms'].values[0]
            for pl, gl in common_keys]

print("-" * 60)
print(f"{'V1 TPOT 范围':>16}  {min(v1_tpots):.1f} ~ {max(v1_tpots):.1f} ms  "
      f"(std={np.std(v1_tpots):.2f}ms)")
print(f"{'V2 TPOT 范围':>16}  {min(v2_tpots):.1f} ~ {max(v2_tpots):.1f} ms  "
      f"(std={np.std(v2_tpots):.2f}ms)")
conclusion = "✅ 抖动收窄" if np.std(v2_tpots) < np.std(v1_tpots) else "⚠️  抖动未收窄，可增加重复次数"
print(f"\n{conclusion}")
print("=" * 60)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import os

# ── 全局字体设置：避免 Colab/Kaggle 中文乱码 ─────────────────
plt.rcParams['font.family'] = 'DejaVu Sans'

# ── 图表保存目录 ─────────────────────────────────────────────
FIG_DIR_V2 = RESULT_DIR_V2 if os.path.exists('/kaggle/working') \
    else "/content/drive/MyDrive/HeteroInfer_Lab/results/exp001/figures"
os.makedirs(FIG_DIR_V2, exist_ok=True)

df_v2 = pd.DataFrame(results_v2)
all_prompt_lens = sorted(df_v2['prompt_len'].unique())
gen_len_list_v2 = sorted(df_v2['gen_len'].unique())

# 颜色方案（与 v2 保持一致）
color_map = {32: '#4C9BE8', 64: '#F5A623', 128: '#7ED321'}

# ═══════════════════════════════════════════════════════════
# 布局设计：2×4 GridSpec
#   ① [0, 0:2]   ② [0, 2:4]
#   ③ [1, 0:2]   ④-A [1, 2]  ④-B [1, 3]
# ═══════════════════════════════════════════════════════════

fig = plt.figure(figsize=(18, 10))
fig.suptitle(
    'Experiment 001 PKV: Gemma 2-2B Profiling (3-run avg + Extended Seq)\n'
    '(Tesla T4, FP16, KV from past_key_values)',
    fontsize=13, fontweight='bold'
)

gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.42, wspace=0.35)

ax1  = fig.add_subplot(gs[0, 0:2])
ax2  = fig.add_subplot(gs[0, 2:4])
ax3  = fig.add_subplot(gs[1, 0:2])
ax4a = fig.add_subplot(gs[1, 2])
ax4b = fig.add_subplot(gs[1, 3])

# ── 图①：TTFT vs Prompt Length ─────────────────────────────
for gl in gen_len_list_v2:
    sub = df_v2[df_v2['gen_len'] == gl].sort_values('prompt_len')
    ax1.plot(sub['prompt_len'], sub['ttft_ms'],
             marker='o', color=color_map.get(gl, 'gray'),
             label=f'gen={gl}', linewidth=2)
ax1.set_xlabel('Prompt Length (tokens)')
ax1.set_ylabel('TTFT (ms)')
ax1.set_title('① TTFT vs Prompt Length (Prefill Latency)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xscale('log', base=2)
ax1.set_xticks(all_prompt_lens)
ax1.xaxis.set_major_formatter(ticker.ScalarFormatter())

# ── 图②：TPOT vs Prompt Length ─────────────────────────────
for gl in gen_len_list_v2:
    sub = df_v2[df_v2['gen_len'] == gl].sort_values('prompt_len')
    ax2.plot(sub['prompt_len'], sub['tpot_ms'],
             marker='s', color=color_map.get(gl, 'gray'),
             label=f'gen={gl}', linewidth=2)
ax2.set_xlabel('Prompt Length (tokens)')
ax2.set_ylabel('TPOT (ms)')
ax2.set_title('② TPOT vs Prompt Length (Decode Latency per Token)')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xscale('log', base=2)
ax2.set_xticks(all_prompt_lens)
ax2.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax2.set_ylim(0, 100)

# ── 图③：Peak Memory vs Prompt Length ──────────────────────
for gl in gen_len_list_v2:
    sub = df_v2[df_v2['gen_len'] == gl].sort_values('prompt_len')
    ax3.plot(sub['prompt_len'], sub['peak_mem_mb'],
             marker='^', color=color_map.get(gl, 'gray'),
             label=f'gen={gl}', linewidth=2)
ax3.axhline(y=16*1024, color='red', linestyle='--', alpha=0.5, label='T4 limit (16GB)')
ax3.set_xlabel('Prompt Length (tokens)')
ax3.set_ylabel('Peak GPU Memory (MB)')
ax3.set_title('③ Peak Memory vs Prompt Length')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_xscale('log', base=2)
ax3.set_xticks(all_prompt_lens)
ax3.xaxis.set_major_formatter(ticker.ScalarFormatter())

# ── 图④-A：KV Cache 理论估算 vs past_key_values 实测 ────────
# 选择最大 gen_len 作为代表组，覆盖 decode 后最终 KV Cache。
# 当前矩阵中通常是 gen=128；写成自动选择，避免后续调整 gen_lengths 后失效。
representative_gen = max(gen_len_list_v2)
sub_rep = df_v2[df_v2['gen_len'] == representative_gen].sort_values('prompt_len')

ax4a.plot(sub_rep['prompt_len'], sub_rep['kv_pkv_final_mb'],
          marker='o', color='#4C9BE8', linestyle='-',
          label='PKV measured', linewidth=2)
ax4a.plot(sub_rep['prompt_len'], sub_rep['kv_est_mb'],
          marker='D', color='#F5A623', linestyle='--',
          label='Formula estimate', linewidth=2)

max_row = sub_rep.loc[sub_rep['prompt_len'].idxmax()]
ratio   = max_row['kv_pkv_final_mb'] / max_row['kv_est_mb']
ax4a.annotate(
    f'~{ratio:.2f}x',
    xy=(max_row['prompt_len'], max_row['kv_pkv_final_mb']),
    xytext=(-45, -30),
    textcoords='offset points',
    fontsize=9, color='#333333',
    arrowprops=dict(arrowstyle='->', color='#666666', lw=1.2),
)

ax4a.set_xlabel('Prompt Length (tokens)')
ax4a.set_ylabel('KV Cache Size (MB)')
ax4a.set_title(f'④-A: KV Cache PKV vs Formula\n(gen={representative_gen})')
ax4a.legend(fontsize=8)
ax4a.grid(True, alpha=0.3)
ax4a.set_xscale('log', base=2)
ax4a.set_xticks(all_prompt_lens)
ax4a.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax4a.tick_params(axis='x', labelrotation=45)

# ── 图④-B：prefill 后 PKV vs decode 后 PKV ─────────────────
# 两条线之间的距离就是 decode 阶段新增 token 带来的 KV Cache 增量。
for col, style, label in [
    ('kv_pkv_prefill_mb', '--', 'PKV after prefill'),
    ('kv_pkv_final_mb', '-', 'PKV after decode'),
]:
    ax4b.plot(sub_rep['prompt_len'], sub_rep[col],
              marker='D', linestyle=style, linewidth=2, label=label)

ax4b.fill_between(
    sub_rep['prompt_len'],
    sub_rep['kv_pkv_prefill_mb'],
    sub_rep['kv_pkv_final_mb'],
    color='#4C9BE8',
    alpha=0.12,
    label='Decode KV growth',
)

ax4b.set_xlabel('Prompt Length (tokens)')
ax4b.set_ylabel('KV Cache Size (MB)')
ax4b.set_title(f'④-B: Prefill vs Final PKV\n(gen={representative_gen})')
ax4b.legend(fontsize=8)
ax4b.grid(True, alpha=0.3)
ax4b.set_xscale('log', base=2)
ax4b.set_xticks(all_prompt_lens)
ax4b.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax4b.tick_params(axis='x', labelrotation=45)

# ── 保存 PKV 图表（不覆盖 v2 / v3）────────────────────────────
fig_path = f"{FIG_DIR_V2}/exp001_profiling_results_pkv.png"
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ PKV 图表已保存：{fig_path}")